# Univariate drift detection and OD flow analysis

This notebook analyzes the ADWIN, KSWIN, and Page-Hinkley detectors near previously identified multivariate drift events. It preserves all detector CSV exports, detector-agreement summaries, and **only** the top-four highest-variance original OD flow figures. The plotting section reads raw traffic matrices: no logarithm, seasonal differencing, or normalization is applied to the plotted values.

**Run in order** from the repository root. Dependencies: Python, NumPy, pandas, matplotlib, and River. Input files must be present in `outputs/`, `results_sem_log/outputs/`, and `datasets/` as configured below.

**Input paths:** The notebook expects the multivariate summary files under `results_sem_log/outputs/` and the processed input CSVs under `outputs/`. Run it from the repository root or update `DATASET_CONFIG` to match your working directory.

**Path compatibility:** The processed seasonal-difference CSVs use the English filenames generated by `main.ipynb`. The detector summary directories under `results_sem_log/outputs/` retain their existing Portuguese names because they are produced by the detector experiments.


## 1. Run univariate detectors and export CSV results

Uses the seasonally differenced, **non-logarithmic** traffic series to evaluate each OD flow around multivariate drift events. Exports detailed detections, summaries, event mappings, and per-flow records to `univariada/<dataset>/`.


In [1]:
import ast
import csv
import json
import math
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from river import drift


# General configuration

MAX_WORKERS = 7

# Seeds used by univariate detectors.
SEEDS = [42]

MULTIVARIATE_WINDOWS = {
    "geant": 60,
    "abilene": 120,
}

# Number of multivariate windows before and after each candidate drift.
FATOR_JANELA = 3

DATASET_CONFIG = {
    "geant": {
        "data_file": "./outputs/geant_seasonal_difference_with_gaps.csv",
        "summary_file": "./results_sem_log/outputs/geant_diferenca_sazonal_sem_log_com_lacunas/geant_summary.csv",
    },
    "abilene": {
        "data_file": "./outputs/abilene_seasonal_difference_with_gaps.csv",
        "summary_file": "./results_sem_log/outputs/abilene_diferenca_sazonal_sem_log_com_lacunas/abilene_summary.csv",
    },
}

# Check paths before submitting tasks to worker processes.
# All paths are relative to the Jupyter working directory.
for dataset_name, paths in DATASET_CONFIG.items():
    for file_kind, file_path in paths.items():
        if not Path(file_path).is_file():
            raise FileNotFoundError(
                f"Missing {file_kind} for {dataset_name}: {Path(file_path).resolve()}\n"
                "Check the Jupyter working directory (Path.cwd()) and the "
                "DATASET_CONFIG paths. The multivariate summary must be "
                "generated before running this notebook."
            )

# Output directory.
OUT_ROOT = Path("./univariada")
OUT_ROOT.mkdir(parents=True, exist_ok=True)


# Univariate detector hyperparameters

HYPERPARAMS = {
    "ADWIN": [
        {"use_library_default": True},
    ],

    "KSWIN": [
        {"use_library_default": True},
    ],

    "PageHinkley": [
        {"use_library_default": True},
    ],
}

# Helper functions

def parse_drift_indices(valor):

    if valor is None:
        return []

    if isinstance(valor, float) and np.isnan(valor):
        return []

    if isinstance(valor, list):
        return [int(x) for x in valor]

    if isinstance(valor, str):
        texto = valor.strip()

        if texto == "" or texto.lower() in {"nan", "none", "null"}:
            return []

        try:
            return [int(x) for x in json.loads(texto)]
        except Exception:
            pass

        try:
            return [int(x) for x in ast.literal_eval(texto)]
        except Exception:
            return []

    return []


def is_missing(val):

    return (
        val is None
        or (isinstance(val, float) and math.isnan(val))
        or not np.isfinite(val)
    )


def criar_detector(detector_name, seed, params):

    params = params.copy()

    if params.pop("use_library_default", False):
        if detector_name == "KSWIN":
            return drift.KSWIN(seed=seed)
        elif detector_name == "ADWIN":
            return drift.ADWIN()
        elif detector_name == "PageHinkley":
            return drift.PageHinkley()

    if detector_name == "ADWIN":
        return drift.ADWIN(**params)

    elif detector_name == "KSWIN":
        params["seed"] = seed
        return drift.KSWIN(**params)

    elif detector_name == "PageHinkley":
        return drift.PageHinkley(**params)

    else:
        raise ValueError(f"Detector desconhecido: {detector_name}")


def carregar_dataset(data_file):

    df = pd.read_csv(
        data_file,
        index_col=0,
        parse_dates=True
    ).sort_index()

    df.index.name = "timestamp"

    return df


def carregar_drifts_multivariados(summary_file, window_escolhida):

    df_summary = pd.read_csv(summary_file)

    sub = df_summary[
        df_summary["window"] == window_escolhida
    ].copy()

    todos_indices = []

    for valor in sub["drift_indices_json"]:
        todos_indices.extend(parse_drift_indices(valor))

    indices_unicos = sorted(set(int(i) for i in todos_indices))

    return indices_unicos, sub


def montar_eventos_multivariados(
    dataset,
    df_data,
    indices_multiv,
    window_multiv,
    fator_janela
):

    T = len(df_data)
    timestamps = df_data.index

    raio = fator_janela * window_multiv

    eventos = []

    for evento_id, drift_idx in enumerate(indices_multiv, start=1):

        if not (0 <= drift_idx < T):
            continue

        inicio = max(0, drift_idx - raio)
        fim = min(T - 1, drift_idx + raio)

        eventos.append({
            "dataset": dataset,
            "evento_multivariado_id": evento_id,
            "multiv_drift_index": int(drift_idx),
            "multiv_drift_timestamp": timestamps[drift_idx],
            "window_multivariada": int(window_multiv),
            "fator_janela": int(fator_janela),
            "segmento_inicio_index": int(inicio),
            "segmento_fim_index": int(fim),
            "segmento_inicio_timestamp": timestamps[inicio],
            "segmento_fim_timestamp": timestamps[fim],
        })

    return pd.DataFrame(eventos)


# Univariate detection

def processar_tarefa_univariada(args):

    (
        dataset,
        evento_multivariado_id,
        multiv_drift_index,
        multiv_drift_timestamp,
        window_multivariada,
        segmento_inicio_index,
        segmento_fim_index,
        timestamps_segmento,
        stream_name,
        stream_values,
        seed,
        detector_name,
        params,
    ) = args

    detector = criar_detector(detector_name, seed, params)

    resultados = []

    for local_pos, valor in enumerate(stream_values):

        global_index = segmento_inicio_index + local_pos
        timestamp_atual = timestamps_segmento[local_pos]

        if is_missing(valor):
            detector = criar_detector(detector_name, seed, params)
            continue

        detector.update(float(valor))

        if detector.drift_detected:
            resultados.append({
                "dataset": dataset,
                "evento_multivariado_id": evento_multivariado_id,

                "multiv_drift_index": multiv_drift_index,
                "multiv_drift_timestamp": str(multiv_drift_timestamp),

                "window_multivariada": window_multivariada,

                "segmento_inicio_index": segmento_inicio_index,
                "segmento_fim_index": segmento_fim_index,

                "stream": stream_name,
                "seed": seed,
                "detector_univariado": detector_name,
                "params": json.dumps(params, sort_keys=True),

                "drift_index_global": global_index,
                "drift_timestamp": str(timestamp_atual),

                "drift_index_local_no_segmento": local_pos,
            })

    return resultados


def montar_tarefas_dataset(dataset, df_data, df_eventos):

    tarefas = []

    detectores_univariados = list(HYPERPARAMS.keys())

    for _, evento in df_eventos.iterrows():

        ini = int(evento["segmento_inicio_index"])
        fim = int(evento["segmento_fim_index"])

        df_segmento = df_data.iloc[ini:fim + 1]
        timestamps_segmento = list(df_segmento.index)

        for stream_name in df_segmento.columns:

            stream_values = df_segmento[stream_name].astype(float).tolist()

            for seed in SEEDS:

                for detector_name in detectores_univariados:

                    for params in HYPERPARAMS[detector_name]:

                        tarefas.append((
                            dataset,
                            int(evento["evento_multivariado_id"]),
                            int(evento["multiv_drift_index"]),
                            evento["multiv_drift_timestamp"],
                            int(evento["window_multivariada"]),
                            ini,
                            fim,
                            timestamps_segmento,
                            stream_name,
                            stream_values,
                            seed,
                            detector_name,
                            params,
                        ))

    return tarefas


# Save CSV results

def salvar_detalhado(df_resultados, out_dir):
    path = out_dir / "univariado_detalhado.csv"
    df_resultados.to_csv(path, index=False)
    return path


def salvar_summary(df_resultados, out_dir):

    if df_resultados.empty:
        df_summary = pd.DataFrame(columns=[
            "dataset",
            "evento_multivariado_id",
            "multiv_drift_index",
            "multiv_drift_timestamp",
            "window_multivariada",
            "stream",
            "seed",
            "detector_univariado",
            "params",
            "n_drifts_univariados",
            "drift_indices_globais_json",
            "drift_timestamps_json",
        ])
    else:
        df_summary = (
            df_resultados
            .groupby([
                "dataset",
                "evento_multivariado_id",
                "multiv_drift_index",
                "multiv_drift_timestamp",
                "window_multivariada",
                "stream",
                "seed",
                "detector_univariado",
                "params",
            ], as_index=False)
            .agg(
                n_drifts_univariados=("drift_index_global", "count"),
                drift_indices_globais_json=(
                    "drift_index_global",
                    lambda s: json.dumps([int(x) for x in s.tolist()])
                ),
                drift_timestamps_json=(
                    "drift_timestamp",
                    lambda s: json.dumps(s.tolist())
                ),
            )
        )

    path = out_dir / "univariado_summary.csv"
    df_summary.to_csv(path, index=False)

    return df_summary, path


def salvar_fluxos_por_evento(df_resultados, out_dir):

    if df_resultados.empty:
        df_fluxos = pd.DataFrame(columns=[
            "dataset",
            "evento_multivariado_id",
            "multiv_drift_index",
            "multiv_drift_timestamp",
            "n_fluxos_com_drift",
            "fluxos_com_drift_json",
        ])
    else:
        df_fluxos = (
            df_resultados
            .groupby([
                "dataset",
                "evento_multivariado_id",
                "multiv_drift_index",
                "multiv_drift_timestamp",
            ], as_index=False)
            .agg(
                n_fluxos_com_drift=("stream", lambda s: len(set(s))),
                fluxos_com_drift_json=(
                    "stream",
                    lambda s: json.dumps(sorted(set(s)))
                ),
            )
        )

    path = out_dir / "fluxos_com_drift_por_evento_multivariado.csv"
    df_fluxos.to_csv(path, index=False)

    return df_fluxos, path


# Run detector analysis

if __name__ == "__main__":

    for dataset in ["geant", "abilene"]:

        print("\n" + "=" * 80)
        print(f"Processing multivariate-guided univariate analysis: {dataset.upper()}")
        print("=" * 80)

        config = DATASET_CONFIG[dataset]

        data_file = config["data_file"]
        summary_file = config["summary_file"]

        window_multiv = MULTIVARIATE_WINDOWS[dataset]

        out_dir = OUT_ROOT / dataset
        out_dir.mkdir(parents=True, exist_ok=True)

        df_data = carregar_dataset(data_file)

        print(f"Dataset loaded: {data_file}")
        print(f"Samples: {len(df_data)}")
        print(f"Flows: {len(df_data.columns)}")

        indices_multiv, df_multiv_filtrado = carregar_drifts_multivariados(
            summary_file=summary_file,
            window_escolhida=window_multiv
        )

        print(f"Multivariate summary: {summary_file}")
        print(f"Multivariate window: {window_multiv}")
        print(f"Unique multivariate drift indices: {len(indices_multiv)}")

        df_multiv_filtrado.to_csv(
            out_dir / "multivariado_summary_filtrado.csv",
            index=False
        )

        df_eventos = montar_eventos_multivariados(
            dataset=dataset,
            df_data=df_data,
            indices_multiv=indices_multiv,
            window_multiv=window_multiv,
            fator_janela=FATOR_JANELA
        )

        df_eventos.to_csv(
            out_dir / "eventos_multivariados_analisados.csv",
            index=False
        )

        print(f"Valid events: {len(df_eventos)}")

        tarefas = montar_tarefas_dataset(
            dataset=dataset,
            df_data=df_data,
            df_eventos=df_eventos
        )

        print(f"Univariate tasks: {len(tarefas)}")

        resultados = []

        with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
            for resultado_tarefa in executor.map(
                processar_tarefa_univariada,
                tarefas,
                chunksize=50
            ):
                resultados.extend(resultado_tarefa)

        df_resultados = pd.DataFrame(resultados)

        print(f"Univariate detections: {len(df_resultados)}")

        path_detalhado = salvar_detalhado(
            df_resultados=df_resultados,
            out_dir=out_dir
        )

        df_summary_uni, path_summary = salvar_summary(
            df_resultados=df_resultados,
            out_dir=out_dir
        )

        df_fluxos, path_fluxos = salvar_fluxos_por_evento(
            df_resultados=df_resultados,
            out_dir=out_dir
        )

        print("\nSaved files:")
        print(f"- {path_detalhado}")
        print(f"- {path_summary}")
        print(f"- {path_fluxos}")
        print(f"- {out_dir / 'eventos_multivariados_analisados.csv'}")
        print(f"- {out_dir / 'multivariado_summary_filtrado.csv'}")

    print("\nProcessing completed.")



Processing multivariate-guided univariate analysis: GEANT
Dataset loaded: ./outputs/geant_seasonal_difference_with_gaps.csv
Samples: 11460
Flows: 462
Multivariate summary: ./results_sem_log/outputs/geant_diferenca_sazonal_sem_log_com_lacunas/geant_summary.csv
Multivariate window: 60
Unique multivariate drift indices: 30
Valid events: 30
Univariate tasks: 41580
Univariate detections: 138734

Saved files:
- univariada/geant/univariado_detalhado.csv
- univariada/geant/univariado_summary.csv
- univariada/geant/fluxos_com_drift_por_evento_multivariado.csv
- univariada/geant/eventos_multivariados_analisados.csv
- univariada/geant/multivariado_summary_filtrado.csv

Processing multivariate-guided univariate analysis: ABILENE
Dataset loaded: ./outputs/abilene_seasonal_difference_with_gaps.csv
Samples: 55872
Flows: 132
Multivariate summary: ./results_sem_log/outputs/abilene_diferenca_sazonal_sem_log_com_lacunas/abilene_summary.csv
Multivariate window: 120
Unique multivariate drift indices: 39
V

## 2. Summarize detector agreement and export CSVs

Aggregates the univariate detection records, matches agreeing detectors (default: exact sample indices), ranks OD flows, and exports per-dataset and combined CSVs to `univariada/analise_resultados/`.


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


# Configuration

INPUTS = {
    "geant": {
        "detalhado": "./univariada/geant/univariado_detalhado.csv",
        "eventos": "./univariada/geant/eventos_multivariados_analisados.csv",
    },
    "abilene": {
        "detalhado": "./univariada/abilene/univariado_detalhado.csv",
        "eventos": "./univariada/abilene/eventos_multivariados_analisados.csv",
    },
}

OUT_DIR = Path("./univariada/analise_resultados")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MARGEM_CONCORDANCIA_INDICES = 0

MIN_DETECTORES_CONCORDANCIA = 2

ORDEM_DETECTORES = ["ADWIN", "KSWIN", "PageHinkley"]


# Helper functions

def lista_unica_ordenada(valores):
    return sorted(set(valores))


def formatar_fluxos_com_indices(df, detector):

    sub = df[df["detector_univariado"] == detector]

    if sub.empty:
        return ""

    partes = []

    for stream, df_stream in sub.groupby("stream"):
        indices = lista_unica_ordenada(
            df_stream["drift_index_global"].astype(int).tolist()
        )

        indices_txt = ", ".join(map(str, indices))
        partes.append(f"{stream} [{indices_txt}]")

    return "; ".join(partes)


def formatar_apenas_fluxos(df, detector):

    sub = df[df["detector_univariado"] == detector]

    if sub.empty:
        return ""

    fluxos = lista_unica_ordenada(sub["stream"].astype(str).tolist())
    return ", ".join(fluxos)


def agrupar_concordancias_evento(
    df_evento,
    margem=0,
    min_detectores=2,
):


    concordancias = []

    if df_evento.empty:
        return pd.DataFrame()

    for stream, df_stream in df_evento.groupby("stream"):

        pontos = (
            df_stream[
                ["drift_index_global", "detector_univariado"]
            ]
            .drop_duplicates()
            .sort_values("drift_index_global")
            .reset_index(drop=True)
        )

        if pontos.empty:
            continue

        grupo_atual = []
        ultimo_indice = None

        for _, row in pontos.iterrows():
            indice = int(row["drift_index_global"])
            detector = row["detector_univariado"]

            if ultimo_indice is None:
                grupo_atual = [(indice, detector)]
                ultimo_indice = indice
                continue

            if indice - ultimo_indice <= margem:
                grupo_atual.append((indice, detector))
            else:
                resultado = resumir_grupo_concordancia(
                    stream=stream,
                    grupo=grupo_atual,
                    min_detectores=min_detectores,
                )

                if resultado is not None:
                    concordancias.append(resultado)

                grupo_atual = [(indice, detector)]

            ultimo_indice = indice

        resultado = resumir_grupo_concordancia(
            stream=stream,
            grupo=grupo_atual,
            min_detectores=min_detectores,
        )

        if resultado is not None:
            concordancias.append(resultado)

    return pd.DataFrame(concordancias)


def resumir_grupo_concordancia(stream, grupo, min_detectores):

    indices = [int(idx) for idx, _ in grupo]
    detectores = sorted(set(det for _, det in grupo))

    if len(detectores) < min_detectores:
        return None

    return {
        "stream": stream,
        "indice_inicio": min(indices),
        "indice_fim": max(indices),
        "indice_centro_aprox": int(round(np.median(indices))),
        "n_detectores_concordando": len(detectores),
        "detectores_concordando": ", ".join(detectores),
        "indices_grupo": ", ".join(map(str, sorted(indices))),
    }


def formatar_concordancias_fluxos(df_concordancias):

    if df_concordancias.empty:
        return ""

    partes = []

    for _, row in df_concordancias.iterrows():
        fluxo = row["stream"]
        indices = row["indices_grupo"]
        detectores = row["detectores_concordando"]

        partes.append(
            f"{fluxo} [{indices}] ({detectores})"
        )

    return "; ".join(partes)


# Per-dataset analysis

tabelas_resumo_multi = []
tabelas_fluxos_ranking = []
tabelas_concordancias_detalhadas = []

for dataset_name, paths in INPUTS.items():

    print("\n" + "=" * 80)
    print(f"Analyzing dataset: {dataset_name.upper()}")
    print("=" * 80)

    path_detalhado = Path(paths["detalhado"])
    path_eventos = Path(paths["eventos"])

    if not path_detalhado.exists():
        print(f"  File not found: {path_detalhado}")
        continue

    if not path_eventos.exists():
        print(f"  File not found: {path_eventos}")
        continue

    df_uni = pd.read_csv(path_detalhado)
    df_eventos = pd.read_csv(path_eventos)

    if df_uni.empty:
        print(f"  No univariate detections found for {dataset_name}.")
        continue

    # Tipos
    df_uni["multiv_drift_index"] = df_uni["multiv_drift_index"].astype(int)
    df_uni["drift_index_global"] = df_uni["drift_index_global"].astype(int)

    df_eventos["multiv_drift_index"] = df_eventos["multiv_drift_index"].astype(int)


    linhas_resumo = []
    concordancias_dataset = []

    for _, evento in df_eventos.iterrows():

        evento_id = int(evento["evento_multivariado_id"])
        indice_multi = int(evento["multiv_drift_index"])

        sub_evento = df_uni[
            df_uni["evento_multivariado_id"] == evento_id
        ].copy()

        n_adwin = int(
            (sub_evento["detector_univariado"] == "ADWIN").sum()
        )
        n_kswin = int(
            (sub_evento["detector_univariado"] == "KSWIN").sum()
        )
        n_page = int(
            (sub_evento["detector_univariado"] == "PageHinkley").sum()
        )

        fluxos_indices_adwin = formatar_fluxos_com_indices(
            sub_evento,
            "ADWIN"
        )
        fluxos_indices_kswin = formatar_fluxos_com_indices(
            sub_evento,
            "KSWIN"
        )
        fluxos_indices_page = formatar_fluxos_com_indices(
            sub_evento,
            "PageHinkley"
        )

        fluxos_adwin = formatar_apenas_fluxos(
            sub_evento,
            "ADWIN"
        )
        fluxos_kswin = formatar_apenas_fluxos(
            sub_evento,
            "KSWIN"
        )
        fluxos_page = formatar_apenas_fluxos(
            sub_evento,
            "PageHinkley"
        )

        df_conc_evento = agrupar_concordancias_evento(
            df_evento=sub_evento,
            margem=MARGEM_CONCORDANCIA_INDICES,
            min_detectores=MIN_DETECTORES_CONCORDANCIA,
        )

        if not df_conc_evento.empty:
            df_conc_evento.insert(0, "dataset", dataset_name)
            df_conc_evento.insert(1, "evento_multivariado_id", evento_id)
            df_conc_evento.insert(2, "multiv_drift_index", indice_multi)

            concordancias_dataset.append(df_conc_evento)

        n_concordancias = len(df_conc_evento)

        if df_conc_evento.empty:
            fluxos_concordancia = ""
            n_fluxos_concordancia = 0
        else:
            fluxos_concordancia = formatar_concordancias_fluxos(
                df_conc_evento
            )
            n_fluxos_concordancia = df_conc_evento["stream"].nunique()

        linhas_resumo.append({
            "dataset": dataset_name,
            "evento_multivariado_id": evento_id,

            "indice_multi": indice_multi,
            "timestamp_multi": evento["multiv_drift_timestamp"],

            "segmento_inicio_index": int(evento["segmento_inicio_index"]),
            "segmento_fim_index": int(evento["segmento_fim_index"]),

            "n_drifts_ADWIN": n_adwin,
            "n_drifts_KSWIN": n_kswin,
            "n_drifts_PageHinkley": n_page,
            "n_drifts_univariados_total": len(sub_evento),

            "fluxos_ADWIN": fluxos_adwin,
            "fluxos_KSWIN": fluxos_kswin,
            "fluxos_PageHinkley": fluxos_page,

            "fluxos_indices_ADWIN": fluxos_indices_adwin,
            "fluxos_indices_KSWIN": fluxos_indices_kswin,
            "fluxos_indices_PageHinkley": fluxos_indices_page,

            "concordam_n_drifts": n_concordancias,
            "concordam_n_fluxos": n_fluxos_concordancia,
            "concordam_fluxos_e_indices": fluxos_concordancia,
        })

    df_resumo_multi = pd.DataFrame(linhas_resumo)

    df_resumo_multi = df_resumo_multi.sort_values(
        ["indice_multi"]
    ).reset_index(drop=True)

    tabelas_resumo_multi.append(df_resumo_multi)

    out_resumo_multi = OUT_DIR / f"{dataset_name}_mapeamento_uni_para_multi.csv"
    df_resumo_multi.to_csv(out_resumo_multi, index=False)

    print(f"   Univariate-to-multivariate table saved to: {out_resumo_multi}")


    if concordancias_dataset:
        df_concordancias_dataset = pd.concat(
            concordancias_dataset,
            ignore_index=True
        )
    else:
        df_concordancias_dataset = pd.DataFrame(columns=[
            "dataset",
            "evento_multivariado_id",
            "multiv_drift_index",
            "stream",
            "indice_inicio",
            "indice_fim",
            "indice_centro_aprox",
            "n_detectores_concordando",
            "detectores_concordando",
            "indices_grupo",
        ])

    tabelas_concordancias_detalhadas.append(df_concordancias_dataset)

    out_concordancias = OUT_DIR / f"{dataset_name}_concordancias_univariadas_detalhadas.csv"
    df_concordancias_dataset.to_csv(out_concordancias, index=False)

    print(f"   Detailed detector agreement saved to: {out_concordancias}")


    df_fluxos_ranking = (
        df_uni
        .groupby("stream", as_index=False)
        .agg(
            total_drifts_univariados=("drift_index_global", "count"),
            eventos_multivariados_distintos=(
                "evento_multivariado_id",
                "nunique"
            ),
            n_drifts_ADWIN=(
                "detector_univariado",
                lambda s: int((s == "ADWIN").sum())
            ),
            n_drifts_KSWIN=(
                "detector_univariado",
                lambda s: int((s == "KSWIN").sum())
            ),
            n_drifts_PageHinkley=(
                "detector_univariado",
                lambda s: int((s == "PageHinkley").sum())
            ),
            detectores_que_marcaram=(
                "detector_univariado",
                lambda s: ", ".join(
                    det for det in ORDEM_DETECTORES
                    if det in set(s)
                )
            ),
        )
        .sort_values(
            ["total_drifts_univariados", "eventos_multivariados_distintos"],
            ascending=[False, False]
        )
        .reset_index(drop=True)
    )

    df_fluxos_ranking.insert(0, "dataset", dataset_name)

    tabelas_fluxos_ranking.append(df_fluxos_ranking)

    out_fluxos = OUT_DIR / f"{dataset_name}_fluxos_com_mais_drifts.csv"
    df_fluxos_ranking.to_csv(out_fluxos, index=False)

    print(f"   Flow ranking saved to: {out_fluxos}")


# Combined results

if tabelas_resumo_multi:
    df_resumo_multi_geral = pd.concat(
        tabelas_resumo_multi,
        ignore_index=True
    )

    path_geral = OUT_DIR / "mapeamento_uni_para_multi_geral.csv"
    df_resumo_multi_geral.to_csv(path_geral, index=False)

    print(f"\n   Combined univariate-to-multivariate table saved to: {path_geral}")

if tabelas_fluxos_ranking:
    df_fluxos_ranking_geral = pd.concat(
        tabelas_fluxos_ranking,
        ignore_index=True
    )

    path_fluxos_geral = OUT_DIR / "fluxos_com_mais_drifts_geral.csv"
    df_fluxos_ranking_geral.to_csv(path_fluxos_geral, index=False)

    print(f"   Combined flow ranking saved to: {path_fluxos_geral}")

if tabelas_concordancias_detalhadas:
    df_concordancias_geral = pd.concat(
        tabelas_concordancias_detalhadas,
        ignore_index=True
    )

    path_conc_geral = OUT_DIR / "concordancias_univariadas_detalhadas_geral.csv"
    df_concordancias_geral.to_csv(path_conc_geral, index=False)

    print(f"   Combined detector agreement saved to: {path_conc_geral}")


# Notebook summary

print("\n" + "=" * 80)
print("Final summary")
print("=" * 80)

if "df_resumo_multi_geral" in globals():
    print("\nUnivariate-to-multivariate mapping:")
    display(df_resumo_multi_geral)

if "df_fluxos_ranking_geral" in globals():
    print("\nFlows with the most drift alarms:")
    display(df_fluxos_ranking_geral)

if "df_concordancias_geral" in globals():
    print("\nDetailed detector agreement:")
    display(df_concordancias_geral)



Analyzing dataset: GEANT
   Univariate-to-multivariate table saved to: univariada/analise_resultados/geant_mapeamento_uni_para_multi.csv
   Detailed detector agreement saved to: univariada/analise_resultados/geant_concordancias_univariadas_detalhadas.csv
   Flow ranking saved to: univariada/analise_resultados/geant_fluxos_com_mais_drifts.csv

Analyzing dataset: ABILENE
   Univariate-to-multivariate table saved to: univariada/analise_resultados/abilene_mapeamento_uni_para_multi.csv
   Detailed detector agreement saved to: univariada/analise_resultados/abilene_concordancias_univariadas_detalhadas.csv
   Flow ranking saved to: univariada/analise_resultados/abilene_fluxos_com_mais_drifts.csv

   Combined univariate-to-multivariate table saved to: univariada/analise_resultados/mapeamento_uni_para_multi_geral.csv
   Combined flow ranking saved to: univariada/analise_resultados/fluxos_com_mais_drifts_geral.csv
   Combined detector agreement saved to: univariada/analise_resultados/concordanci

,dataset,evento_multivariado_id,indice_multi,timestamp_multi,segmento_inicio_index,segmento_fim_index,n_drifts_ADWIN,n_drifts_KSWIN,n_drifts_PageHinkley,n_drifts_univariados_total,fluxos_ADWIN,fluxos_KSWIN,fluxos_PageHinkley,fluxos_indices_ADWIN,fluxos_indices_KSWIN,fluxos_indices_PageHinkley,concordam_n_drifts,concordam_n_fluxos,concordam_fluxos_e_indices
0,geant,1,3437,2005-06-09 10:15:00,3257,3617,475,1105,3630,5210,"at1.at_ch1.ch, at1.at_de1.de, at1.at_es1.es, a...","at1.at_be1.be, at1.at_ch1.ch, at1.at_de1.de, a...","at1.at_be1.be, at1.at_ch1.ch, at1.at_de1.de, a...","at1.at_ch1.ch [3472]; at1.at_de1.de [3344, 353...","at1.at_be1.be [3408, 3552]; at1.at_ch1.ch [338...","at1.at_be1.be [3310, 3340, 3374, 3404, 3443, 3...",29,28,"at1.at_ie1.ie [3483, 3483] (KSWIN, PageHinkley..."
1,geant,2,7251,2005-07-19 03:45:00,7071,7431,544,1140,3420,5104,"at1.at_ch1.ch, at1.at_de1.de, at1.at_fr1.fr, a...","at1.at_be1.be, at1.at_ch1.ch, at1.at_de1.de, a...","at1.at_be1.be, at1.at_ch1.ch, at1.at_de1.de, a...","at1.at_ch1.ch [7262]; at1.at_de1.de [7294, 735...","at1.at_be1.be [7171, 7332]; at1.at_ch1.ch [720...","at1.at_be1.be [7110, 7140, 7175, 7207, 7268, 7...",40,40,"at1.at_fr1.fr [7179, 7179] (KSWIN, PageHinkley..."
2,geant,3,9259,2005-08-09 01:45:00,9079,9439,465,1138,3266,4869,"at1.at_ch1.ch, at1.at_de1.de, at1.at_es1.es, a...","at1.at_be1.be, at1.at_ch1.ch, at1.at_de1.de, a...","at1.at_be1.be, at1.at_ch1.ch, at1.at_de1.de, a...",at1.at_ch1.ch [9270]; at1.at_de1.de [9366]; at...,"at1.at_be1.be [9199]; at1.at_ch1.ch [9183, 928...","at1.at_be1.be [9112, 9165, 9195, 9252, 9307, 9...",36,35,"at1.at_il1.il [9408, 9408] (KSWIN, PageHinkley..."
3,geant,4,9357,2005-08-10 02:15:00,9177,9537,465,1122,3365,4952,"at1.at_be1.be, at1.at_ch1.ch, at1.at_de1.de, a...","at1.at_be1.be, at1.at_ch1.ch, at1.at_de1.de, a...","at1.at_be1.be, at1.at_ch1.ch, at1.at_de1.de, a...","at1.at_be1.be [9496]; at1.at_ch1.ch [9272, 952...","at1.at_be1.be [9276, 9456]; at1.at_ch1.ch [928...","at1.at_be1.be [9209, 9252, 9307, 9337, 9379, 9...",32,32,"be1.be_uk1.uk [9510, 9510] (KSWIN, PageHinkley..."
4,geant,5,9410,2005-08-10 15:30:00,9230,9590,434,1096,3333,4863,"at1.at_be1.be, at1.at_ch1.ch, at1.at_de1.de, a...","at1.at_be1.be, at1.at_ch1.ch, at1.at_de1.de, a...","at1.at_be1.be, at1.at_ch1.ch, at1.at_de1.de, a...","at1.at_be1.be [9549]; at1.at_ch1.ch [9421, 958...","at1.at_be1.be [9364, 9505]; at1.at_ch1.ch [940...","at1.at_be1.be [9259, 9307, 9337, 9379, 9417, 9...",44,42,"be1.be_cz1.cz [9589, 9589] (KSWIN, PageHinkley..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,abilene,35,53084,2004-09-01 07:40:00,52724,53444,488,764,1952,3204,"ATLAM5_ATLAng, ATLAM5_CHINng, ATLAM5_DNVRng, A...","ATLAM5_ATLAng, ATLAM5_CHINng, ATLAM5_DNVRng, A...","ATLAM5_ATLAng, ATLAM5_CHINng, ATLAM5_IPLSng, A...","ATLAM5_ATLAng [52915, 53171, 53427]; ATLAM5_CH...","ATLAM5_ATLAng [52853, 53002, 53103, 53204, 533...","ATLAM5_ATLAng [52753, 52875, 53141, 53248, 532...",27,25,"ATLAng_IPLSng [53063, 53063] (KSWIN, PageHinkl..."
65,abilene,36,54894,2004-09-07 14:30:00,54534,55254,541,777,2026,3344,"ATLAM5_ATLAng, ATLAM5_CHINng, ATLAM5_HSTNng, A...","ATLAM5_ATLAng, ATLAM5_CHINng, ATLAM5_DNVRng, A...","ATLAM5_ATLAng, ATLAM5_CHINng, ATLAM5_HSTNng, A...","ATLAM5_ATLAng [54725, 55141]; ATLAM5_CHINng [5...","ATLAM5_ATLAng [54633, 54739, 54871, 55039, 551...","ATLAM5_ATLAng [54678, 54712, 54844, 54874, 551...",38,32,"ATLAng_DNVRng [55152, 55152] (KSWIN, PageHinkl..."
66,abilene,37,55443,2004-09-09 12:15:00,55083,55803,470,749,1995,3214,"ATLAM5_ATLAng, ATLAM5_CHINng, ATLAM5_DNVRng, A...","ATLAM5_ATLAng, ATLAM5_CHINng, ATLAM5_DNVRng, A...","ATLAM5_ATLAng, ATLAM5_CHINng, ATLAM5_DNVRng, A...","ATLAM5_ATLAng [55434]; ATLAM5_CHINng [55338, 5...","ATLAM5_ATLAng [55205, 55305, 55432, 55735]; AT...","ATLAM5_ATLAng [55251, 55303, 55803]; ATLAM5_CH...",34,33,"ATLAng_HSTNng [55397, 55397] (KSWIN, PageHinkl..."
67,abilene,38,55550,2004-09-09 21:10:00,55190,55871,449,718,1924,3091,"AT


Flows with the most drift alarms:


,dataset,stream,total_drifts_univariados,eventos_multivariados_distintos,n_drifts_ADWIN,n_drifts_KSWIN,n_drifts_PageHinkley,detectores_que_marcaram
0,geant,gr1.gr_de1.de,549,30,110,88,351,"ADWIN, KSWIN, PageHinkley"
1,geant,ch1.ch_at1.at,548,30,114,84,350,"ADWIN, KSWIN, PageHinkley"
2,geant,ch1.ch_es1.es,543,30,105,87,351,"ADWIN, KSWIN, PageHinkley"
3,geant,es1.es_de1.de,536,30,99,86,351,"ADWIN, KSWIN, PageHinkley"
4,geant,at1.at_se1.se,529,30,95,85,349,"ADWIN, KSWIN, PageHinkley"
...,...,...,...,...,...,...,...,...
586,abilene,HSTNng_ATLAM5,255,31,40,107,108,"ADWIN, KSWIN, PageHinkley"
587,abilene,DNVRng_ATLAM5,220,31,29,153,38,"ADWIN, KSWIN, PageHinkley"
588,abilene,ATLAM5_DNVRng,209,37,26,171,12,"ADWIN, KSWIN, PageHinkley"
589,abilene,ATLAM5_SNVAng,155,38,5,147,3,"ADWIN, KSWIN, PageHinkley"



Detailed detector agreement:


,dataset,evento_multivariado_id,multiv_drift_index,stream,indice_inicio,indice_fim,indice_centro_aprox,n_detectores_concordando,detectores_concordando,indices_grupo
0,geant,1,3437,at1.at_ie1.ie,3483,3483,3483,2,"KSWIN, PageHinkley","3483, 3483"
1,geant,1,3437,at1.at_nl1.nl,3611,3611,3611,2,"KSWIN, PageHinkley","3611, 3611"
2,geant,1,3437,be1.be_es1.es,3582,3582,3582,2,"KSWIN, PageHinkley","3582, 3582"
3,geant,1,3437,de1.de_nl1.nl,3580,3580,3580,2,"KSWIN, PageHinkley","3580, 3580"
4,geant,1,3437,de1.de_pt1.pt,3400,3400,3400,2,"KSWIN, PageHinkley","3400, 3400"
...,...,...,...,...,...,...,...,...,...,...
2100,abilene,39,55564,NYCMng_LOSAng,55427,55427,55427,2,"ADWIN, PageHinkley","55427, 55427"
2101,abilene,39,55564,NYCMng_WASHng,55503,55503,55503,2,"KSWIN, PageHinkley","55503, 55503"
2102,abilene,39,55564,STTLng_IPLSng,55503,55503,55503,2,"KSWIN, PageHinkley","55503, 55503"
2103,abilene,39,55564,STTLng_IPLSng,55713,55713,55713,2,"KSWIN, PageHinkley","55713, 55713"


## 3. Plot the four highest-variance **original** OD flows

Selects up to four original OD series by within-event variance, following the existing agreement-based candidate selection. Uses the original CSVs in `datasets/`, without log transformation, seasonal differencing, or normalization. Retains the figure export and its accompanying CSV summary.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# Configuration

TOP_N_FLUXOS = 4

DATA_FILES_ORIGINAIS = {
    "geant": "./datasets/geant_demand_matrix.csv",
    "abilene": "./datasets/abilene_demand_matrix.csv",
}

ARQUIVOS_ANALISE = {
    "geant": {
        "eventos": "./univariada/geant/eventos_multivariados_analisados.csv",
        "concordancias": "./univariada/analise_resultados/geant_concordancias_univariadas_detalhadas.csv",
    },
    "abilene": {
        "eventos": "./univariada/abilene/eventos_multivariados_analisados.csv",
        "concordancias": "./univariada/analise_resultados/abilene_concordancias_univariadas_detalhadas.csv",
    },
}

OUT_DIR = Path(
    f"./univariada/plots_top{TOP_N_FLUXOS}_fluxos_dado_original_sem_transformacao"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

NORMALIZAR = False

# Mostra legenda
MOSTRAR_LEGENDA = True

ALPHA_LINHAS = 0.85

DATASETS_FILTRAR = None


# Helper functions

def parse_indices_grupo(valor):

    if pd.isna(valor):
        return []

    texto = str(valor).strip()

    if texto == "":
        return []

    return [int(x.strip()) for x in texto.split(",") if x.strip()]


def nome_seguro(texto):
    return (
        str(texto)
        .replace("GÉANT", "GEANT")
        .replace("É", "E")
        .replace("é", "e")
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "-")
    )


def carregar_serie_dataset(dataset):

    caminho = DATA_FILES_ORIGINAIS[dataset]

    df = pd.read_csv(caminho)

    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.set_index("timestamp")
    else:
        df.index = pd.to_datetime(df.index)

    df = df.replace([np.inf, -np.inf], np.nan)

    df = df.sort_index()
    df.index.name = "timestamp"

    return df


def selecionar_top_fluxos_maior_variancia_original(
    trecho_df,
    fluxos_concordantes,
    top_n=TOP_N_FLUXOS,
):

    scores = []

    for fluxo in fluxos_concordantes:
        if fluxo not in trecho_df.columns:
            continue

        serie = trecho_df[fluxo].astype(float)

        score = serie.var(skipna=True)

        if pd.isna(score):
            continue

        scores.append({
            "stream": fluxo,
            "score_variancia_original": score,
        })

    df_scores = pd.DataFrame(scores)

    if df_scores.empty:
        return [], df_scores

    df_scores = df_scores.sort_values(
        "score_variancia_original",
        ascending=False
    ).reset_index(drop=True)

    top_fluxos = df_scores.head(top_n)["stream"].tolist()

    return top_fluxos, df_scores


# Plot a single event

def obter_posicao_por_timestamp(df_series, timestamp):

    timestamp = pd.to_datetime(timestamp)

    if timestamp in df_series.index:
        return df_series.index.get_loc(timestamp)

    pos = df_series.index.get_indexer([timestamp], method="nearest")[0]

    if pos == -1:
        return None

    return int(pos)

def salvar_plot_top_fluxos_evento_original_sem_transformacao(
    dataset,
    evento_multivariado_id,
    df_series,
    df_eventos,
    df_conc,
    out_dir,
    top_n=TOP_N_FLUXOS,
    mostrar_legenda=True,
    alpha_linhas=0.85,
):
    dataset = dataset.lower()

    evento = df_eventos[
        df_eventos["evento_multivariado_id"] == evento_multivariado_id
    ]

    if evento.empty:
        return None

    evento = evento.iloc[0]

    idx_multi_detector = int(evento["multiv_drift_index"])

    # Timestamp salvo no CSV de eventos
    ts_multi_csv = pd.to_datetime(evento["multiv_drift_timestamp"])

    pos_multi = obter_posicao_por_timestamp(df_series, ts_multi_csv)

    if pos_multi is None:
        return {
            "dataset": dataset,
            "evento_multivariado_id": evento_multivariado_id,
            "status": "timestamp_drift_fora_do_dataset_original",
            "arquivo": "",
            "n_fluxos_concordantes": 0,
            "n_fluxos_plotados": 0,
        }

    ts_multi = df_series.index[pos_multi]

    ini_seg_detector = int(evento["segmento_inicio_index"])
    fim_seg_detector = int(evento["segmento_fim_index"])

    ini_seg = pos_multi - (idx_multi_detector - ini_seg_detector)
    fim_seg = pos_multi + (fim_seg_detector - idx_multi_detector)

    ini_seg = max(0, int(ini_seg))
    fim_seg = min(len(df_series) - 1, int(fim_seg))

    if ini_seg > fim_seg:
        margem = 100
        ini_seg = max(0, pos_multi - margem)
        fim_seg = min(len(df_series) - 1, pos_multi + margem)

    conc_evento = df_conc[
        df_conc["evento_multivariado_id"] == evento_multivariado_id
    ].copy()

    if conc_evento.empty:
        return {
            "dataset": dataset,
            "evento_multivariado_id": evento_multivariado_id,
            "status": "sem_concordancia",
            "arquivo": "",
            "n_fluxos_concordantes": 0,
            "n_fluxos_plotados": 0,
        }

    fluxos_concordantes = conc_evento["stream"].drop_duplicates().tolist()

    trecho_df = df_series.iloc[ini_seg:fim_seg + 1].copy()

    print("\nEvent:", evento_multivariado_id)
    print("ts_multi_csv:", ts_multi_csv)
    print("ts_multi_usado:", ts_multi)
    print("idx_multi_detector:", idx_multi_detector)
    print("pos_multi_usado_no_original:", pos_multi)
    print("ini_seg_detector:", ini_seg_detector)
    print("fim_seg_detector:", fim_seg_detector)
    print("ini_seg_usado:", ini_seg, "->", df_series.index[ini_seg])
    print("fim_seg_usado:", fim_seg, "->", df_series.index[fim_seg])
    print("segment start:", trecho_df.index.min())
    print("segment end:", trecho_df.index.max())
    print(
        "drift dentro do trecho?",
        trecho_df.index.min() <= ts_multi <= trecho_df.index.max()
    )

    top_fluxos, df_scores = selecionar_top_fluxos_maior_variancia_original(
        trecho_df=trecho_df,
        fluxos_concordantes=fluxos_concordantes,
        top_n=top_n,
    )

    if len(top_fluxos) == 0:
        return {
            "dataset": dataset,
            "evento_multivariado_id": evento_multivariado_id,
            "status": "sem_fluxo_valido",
            "arquivo": "",
            "n_fluxos_concordantes": len(fluxos_concordantes),
            "n_fluxos_plotados": 0,
        }

    fig, ax = plt.subplots(figsize=(16, 7))

    for fluxo in top_fluxos:
        serie_fluxo = trecho_df[fluxo].astype(float).copy()

        ax.plot(
            trecho_df.index,
            serie_fluxo.values,
            linewidth=1.3,
            alpha=alpha_linhas,
            label=fluxo
        )

    # Marca drift multivariado
    ax.axvline(
        ts_multi,
        color="black",
        linestyle="--",
        linewidth=2.0,
        label="Multivariate drift"
    )

    conc_top = conc_evento[
        conc_evento["stream"].isin(top_fluxos)
    ].copy()

    for _, row in conc_top.iterrows():

        fluxo = row["stream"]

        if fluxo not in trecho_df.columns:
            continue

        indices_grupo = parse_indices_grupo(row["indices_grupo"])

        for idx_uni in indices_grupo:
            pos_uni = pos_multi + (idx_uni - idx_multi_detector)

            if pos_uni < 0 or pos_uni >= len(df_series):
                continue

            if pos_uni < ini_seg or pos_uni > fim_seg:
                continue

            ts_uni = df_series.index[pos_uni]

            if ts_uni not in trecho_df.index:
                continue

            # Mantido comentado.
            # Se quiser marcar os alarmes univariados, descomente.
            
            # ax.scatter(
            #     ts_uni,
            #     valor_y,
            #     s=45,
            #     zorder=5,
            #     marker="o",
            # )

    # Estilo
    ax.set_title(
        f"{dataset.upper()} — Top {top_n} highest-variance original OD flows"
        f" | Timestamp: {ts_multi}"
    )

    ax.set_xlabel("Time")
    ax.set_ylabel("Original OD flow traffic")

    ax.grid(True, alpha=0.3)

    if mostrar_legenda:
        ax.legend(
            loc="best",
            fontsize=8,
            ncol=2
        )

    plt.tight_layout()

    # Salvar
    pasta_dataset = out_dir / nome_seguro(dataset)
    pasta_dataset.mkdir(parents=True, exist_ok=True)

    out_path = (
        pasta_dataset
        / f"{nome_seguro(dataset)}_evento_{evento_multivariado_id:04d}"
          f"_top{top_n}_fluxos_original_sem_transformacao.png"
    )

    fig.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    return {
        "dataset": dataset,
        "evento_multivariado_id": evento_multivariado_id,
        "indice_multi_detector": idx_multi_detector,
        "indice_multi_usado_no_original": pos_multi,
        "timestamp_multi_csv": ts_multi_csv,
        "timestamp_multi": ts_multi,
        "segmento_inicio_index_detector": ini_seg_detector,
        "segmento_fim_index_detector": fim_seg_detector,
        "segmento_inicio_index_usado": ini_seg,
        "segmento_fim_index_usado": fim_seg,
        "n_fluxos_concordantes": len(fluxos_concordantes),
        "n_fluxos_plotados": len(top_fluxos),
        "top_fluxos": ", ".join(top_fluxos),
        "criterio_selecao": "maior_variancia_no_trecho_original",
        "normalizacao": False,
        "log": False,
        "diferenca_sazonal": False,
        "transformacao": "nenhuma",
        "tipo_dado": "dataset_original",
        "status": "ok",
        "arquivo": str(out_path),
    }


# Generate figures for all events

datasets_para_processar = list(DATA_FILES_ORIGINAIS.keys())

if DATASETS_FILTRAR is not None:
    datasets_para_processar = [
        d for d in datasets_para_processar
        if d in DATASETS_FILTRAR
    ]

linhas_resumo = []

for dataset in datasets_para_processar:

    print("\n" + "=" * 80)
    print(f"Processing original dataset: {dataset.upper()}")
    print("=" * 80)

    df_series = carregar_serie_dataset(dataset)

    df_eventos = pd.read_csv(
        ARQUIVOS_ANALISE[dataset]["eventos"]
    )

    df_conc = pd.read_csv(
        ARQUIVOS_ANALISE[dataset]["concordancias"]
    )

    eventos_ids = sorted(
        df_eventos["evento_multivariado_id"].astype(int).unique()
    )

    print(f"Events found: {len(eventos_ids)}")
    print(f"Original dataset shape: {df_series.shape}")
    print(f"Number of numerical flows: {df_series.select_dtypes(include=[np.number]).shape[1]}")

    for evento_id in eventos_ids:

        resultado = salvar_plot_top_fluxos_evento_original_sem_transformacao(
            dataset=dataset,
            evento_multivariado_id=evento_id,
            df_series=df_series,
            df_eventos=df_eventos,
            df_conc=df_conc,
            out_dir=OUT_DIR,
            top_n=TOP_N_FLUXOS,
            mostrar_legenda=MOSTRAR_LEGENDA,
            alpha_linhas=ALPHA_LINHAS,
        )

        if resultado is not None:
            linhas_resumo.append(resultado)

    print(f"Completed: {dataset.upper()}")


df_resumo_plots = pd.DataFrame(linhas_resumo)

path_resumo = (
    OUT_DIR
    / f"resumo_plots_top{TOP_N_FLUXOS}_fluxos_original_sem_transformacao.csv"
)

df_resumo_plots.to_csv(path_resumo, index=False)

print("\nProcessing completed.")
print(f"Output directory: {OUT_DIR}")
print(f"Summary saved to: {path_resumo}")

display(df_resumo_plots)



Processing original dataset: GEANT
Events found: 30
Original dataset shape: (11460, 462)
Number of numerical flows: 462

Event: 1
ts_multi_csv: 2005-06-09 10:15:00
ts_multi_usado: 2005-06-09 10:15:00
idx_multi_detector: 3437
pos_multi_usado_no_original: 3437
ini_seg_detector: 3257
fim_seg_detector: 3617
ini_seg_usado: 3257 -> 2005-06-07 13:15:00
fim_seg_usado: 3617 -> 2005-06-11 07:15:00
segment start: 2005-06-07 13:15:00
segment end: 2005-06-11 07:15:00
drift dentro do trecho? True

Event: 2
ts_multi_csv: 2005-07-19 03:45:00
ts_multi_usado: 2005-07-19 03:45:00
idx_multi_detector: 7251
pos_multi_usado_no_original: 7251
ini_seg_detector: 7071
fim_seg_detector: 7431
ini_seg_usado: 7071 -> 2005-07-17 06:45:00
fim_seg_usado: 7431 -> 2005-07-21 00:45:00
segment start: 2005-07-17 06:45:00
segment end: 2005-07-21 00:45:00
drift dentro do trecho? True

Event: 3
ts_multi_csv: 2005-08-09 01:45:00
ts_multi_usado: 2005-08-09 01:45:00
idx_multi_detector: 9259
pos_multi_usado_no_original: 9259
ini_

,dataset,evento_multivariado_id,indice_multi_detector,indice_multi_usado_no_original,timestamp_multi_csv,timestamp_multi,segmento_inicio_index_detector,segmento_fim_index_detector,segmento_inicio_index_usado,segmento_fim_index_usado,...,n_fluxos_plotados,top_fluxos,criterio_selecao,normalizacao,log,diferenca_sazonal,transformacao,tipo_dado,status,arquivo
0,geant,1,3437,3437,2005-06-09 10:15:00,2005-06-09 10:15:00,3257,3617,3257,3617,...,4,"hu1.hu_de1.de, hr1.hr_de1.de, hu1.hu_se1.se, n...",maior_variancia_no_trecho_original,False,False,False,nenhuma,dataset_original,ok,univariada/plots_top4_fluxos_dado_original_sem...
1,geant,2,7251,7251,2005-07-19 03:45:00,2005-07-19 03:45:00,7071,7431,7071,7431,...,4,"ch1.ch_se1.se, gr1.gr_de1.de, hr1.hr_de1.de, a...",maior_variancia_no_trecho_original,False,False,False,nenhuma,dataset_original,ok,univariada/plots_top4_fluxos_dado_original_sem...
2,geant,3,9259,9259,2005-08-09 01:45:00,2005-08-09 01:45:00,9079,9439,9079,9439,...,4,"ny1.ny_uk1.uk, at1.at_si1.si, at1.at_il1.il, s...",maior_variancia_no_trecho_original,False,False,False,nenhuma,dataset_original,ok,univariada/plots_top4_fluxos_dado_original_sem...
3,geant,4,9357,9357,2005-08-10 02:15:00,2005-08-10 02:15:00,9177,9537,9177,9537,...,4,"ch1.ch_se1.se, ch1.ch_es1.es, ny1.ny_de1.de, d...",maior_variancia_no_trecho_original,False,False,False,nenhuma,dataset_original,ok,univariada/plots_top4_fluxos_dado_original_sem...
4,geant,5,9410,9410,2005-08-10 15:30:00,2005-08-10 15:30:00,9230,9590,9230,9590,...,4,"ch1.ch_se1.se, ch1.ch_at1.at, ch1.ch_es1.es, s...",maior_variancia_no_trecho_original,False,False,False,nenhuma,dataset_original,ok,univariada/plots_top4_fluxos_dado_original_sem...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,abilene,35,53084,45308,2004-09-01 07:40:00,2004-09-01 07:40:00,52724,53444,44948,45668,...,4,"LOSAng_CHINng, WASHng_CHINng, CHINng_ATLAng, S...",maior_variancia_no_trecho_original,False,False,False,nenhuma,dataset_original,ok,univariada/plots_top4_fluxos_dado_original_sem...
65,abilene,36,54894,47118,2004-09-07 14:30:00,2004-09-07 14:30:00,54534,55254,46758,47478,...,4,"STTLng_IPLSng, STTLng_WASHng, WASHng_NYCMng, S...",maior_variancia_no_trecho_original,False,False,False,nenhuma,dataset_original,ok,univariada/plots_top4_fluxos_dado_original_sem...
66,abilene,37,55443,47667,2004-09-09 12:15:00,2004-09-09 12:15:00,55083,55803,47307,48027,...,4,"CHINng_LOSAng, LOSAng_CHINng, STTLng_CHINng, S...",maior_variancia_no_trecho_original,False,False,False,nenhuma,dataset_original,ok,univariada/plots_top4_fluxos_dado_original_sem...
67,abilene,38,55550,47774,2004-09-09 21:10:00,2004-09-09 21:10:00,55190,55871,47414,48095,...,4,"STTLng_WASHng, WASHng_NYCMng, CHINng_ATLAng, N...",maior_variancia_no_trecho_original,False,False,False,nenhuma,dataset_original,ok,univariada/plots_top4_fluxos_dado_original_sem...
